# LR vs RF vs XGBoost — Head-to-Head on Unseen Candles

Compare all three models on forward-test data using **pre-trained models from `models/`** (no re-training).

- **LR:** `logistic_v1.joblib` + `scaler_v1.joblib` + `feature_cols_v1.joblib`
- **RF:** `rf_v1.joblib` + `rf_scaler_v1.joblib` + `rf_feature_cols_v1.joblib`
- **XGB:** `xgb_calibrator_v1.joblib` + `xgb_scaler_v1.joblib` + `xgb_feature_cols_v1.joblib`

All tested on newest candles from DB (never seen during training).


In [ ]:
import json
import random
import sqlite3
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from technicals import CandleRecord, IndicatorSnapshot, compute_all
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../data/latest_features.jsonl")
DB_PATH = Path("../data/collection.db")
MODELS_DIR = Path("../models")
MAX_BID = 0.85
WARM_UP = 21

## 1. Load pre-trained models from `models/`

In [ ]:
# LR
lr_model = joblib.load(MODELS_DIR / "logistic_v1.joblib")
lr_scaler = joblib.load(MODELS_DIR / "scaler_v1.joblib")
lr_feat_cols = joblib.load(MODELS_DIR / "feature_cols_v1.joblib")

# RF
rf_model = joblib.load(MODELS_DIR / "rf_v1.joblib")
rf_scaler = joblib.load(MODELS_DIR / "rf_scaler_v1.joblib")
rf_feat_cols = joblib.load(MODELS_DIR / "rf_feature_cols_v1.joblib")

# XGBoost (calibrated)
xgb_model = joblib.load(MODELS_DIR / "xgb_calibrator_v1.joblib")
xgb_scaler = joblib.load(MODELS_DIR / "xgb_scaler_v1.joblib")
xgb_feat_cols = joblib.load(MODELS_DIR / "xgb_feature_cols_v1.joblib")

print(f"LR:  {len(lr_feat_cols)} features, {lr_model.coef_.size} params")
print(f"RF:  {len(rf_feat_cols)} features, {sum(e.tree_.node_count for e in rf_model.estimators_) * 2:,} params")
print(f"XGB: {len(xgb_feat_cols)} features (calibrated)")

# All features needed for compute_all
all_feat_cols = sorted(set(lr_feat_cols) | set(rf_feat_cols) | set(xgb_feat_cols))
print(f"\nUnion of all features: {len(all_feat_cols)}")

## 2. Find training data cutoff timestamp

In [ ]:
# Load training data only to find the cutoff timestamp
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df_train = pd.DataFrame(rows)
max_train_ts = df_train["timestamp"].max()
n_train_candles = df_train["candle_id"].nunique()

print(f"Training data: {n_train_candles} candles")
print(f"Cutoff timestamp: {max_train_ts}")

del df_train  # free memory

## 3. Load and compute features for new candles

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
candles_df = pd.read_sql(
    f"SELECT * FROM candles WHERE start_time > {max_train_ts} ORDER BY start_time",
    conn,
)
snaps_df = (
    pd.read_sql(
        "SELECT * FROM snapshots WHERE candle_id IN ({}) ORDER BY candle_id, timestamp".format(
            ",".join(f"'{cid}'" for cid in candles_df["candle_id"])
        ),
        conn,
    )
    if len(candles_df) > 0
    else pd.DataFrame()
)
prior_candles_df = pd.read_sql(
    f"SELECT * FROM candles WHERE start_time <= {max_train_ts} ORDER BY start_time DESC LIMIT {WARM_UP}",
    conn,
)
conn.close()

prior_candles_df = prior_candles_df.sort_values("start_time")
prior_candles = []
for _, cr in prior_candles_df.iterrows():
    prior_candles.append(
        CandleRecord(
            candle_id=cr["candle_id"],
            start_time=cr["start_time"],
            end_time=cr["end_time"],
            open=cr["open"],
            high=cr["high"],
            low=cr["low"],
            close=cr["close"],
            volume=cr["volume"],
            outcome=cr["outcome"],
            final_ret=cr["final_ret"],
        )
    )

all_rows = []
for _, cr in tqdm(candles_df.iterrows(), total=len(candles_df), desc="Computing features"):
    cid = cr["candle_id"]
    candle = CandleRecord(
        candle_id=cid,
        start_time=cr["start_time"],
        end_time=cr["end_time"],
        open=cr["open"],
        high=cr["high"],
        low=cr["low"],
        close=cr["close"],
        volume=cr["volume"],
        outcome=cr["outcome"],
        final_ret=cr["final_ret"],
    )
    snap_rows = snaps_df[snaps_df["candle_id"] == cid]
    if len(snap_rows) < 5:
        prior_candles.append(candle)
        continue
    snapshots = []
    for _, s in snap_rows.iterrows():
        ob = json.loads(s["orderbook_json"])
        snapshots.append(
            IndicatorSnapshot(
                candle_id=cid,
                timestamp=s["timestamp"],
                elapsed_pct=s["elapsed_pct"],
                btc_price=s["btc_price"],
                btc_bid=s["btc_bid"],
                btc_ask=s["btc_ask"],
                up_bids=[ob["up_bids"][0]] if ob.get("up_bids") else [],
                up_asks=[ob["up_asks"][0]] if ob.get("up_asks") else [],
                down_bids=[ob["down_bids"][0]] if ob.get("down_bids") else [],
                down_asks=[ob["down_asks"][0]] if ob.get("down_asks") else [],
                market_volume=s["market_volume"],
            )
        )
    for si in range(len(snapshots)):
        indicators = compute_all(prior_candles, candle.open, snapshots[: si + 1])
        snap = snapshots[si]
        row = {
            "candle_id": cid,
            "timestamp": snap.timestamp,
            "elapsed_pct": snap.elapsed_pct,
            "btc_price": snap.btc_price,
            "up_best_ask": snap.up_asks[0][0] if snap.up_asks else None,
            "down_best_ask": snap.down_asks[0][0] if snap.down_asks else None,
            **indicators,
            "outcome": candle.outcome,
        }
        all_rows.append(row)
    prior_candles.append(candle)

df_eval = pd.DataFrame(all_rows)
df_eval["target"] = (df_eval["outcome"] == "UP").astype(int)
for col in all_feat_cols:
    if col not in df_eval.columns:
        df_eval[col] = 0.0
df_eval[all_feat_cols] = df_eval[all_feat_cols].fillna(0.0)

print(f"\nNew candles: {df_eval['candle_id'].nunique()}")
print(f"Rows: {len(df_eval):,}")

## 4. Build per-candle predictions for all 3 models

In [ ]:
all_cd = []

for cid in df_eval["candle_id"].unique():
    snap_rows = df_eval[df_eval["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue
    truth = int(snap_rows["target"].iloc[0])

    X_lr = lr_scaler.transform(snap_rows[lr_feat_cols].values)
    lr_probs = lr_model.predict_proba(X_lr)[:, 1]

    X_rf = rf_scaler.transform(snap_rows[rf_feat_cols].values)
    rf_probs = rf_model.predict_proba(X_rf)[:, 1]

    X_xgb = xgb_scaler.transform(snap_rows[xgb_feat_cols].values)
    xgb_probs = xgb_model.predict_proba(X_xgb)[:, 1]

    up_asks = snap_rows["up_best_ask"].values
    down_asks = snap_rows["down_best_ask"].values
    elapsed = snap_rows["elapsed_pct"].values

    sd = [
        {
            "tick": i,
            "elapsed_pct": elapsed[i],
            "lr_pred": int(lr_probs[i] >= 0.5),
            "lr_prob": float(lr_probs[i]),
            "rf_pred": int(rf_probs[i] >= 0.5),
            "rf_prob": float(rf_probs[i]),
            "xgb_pred": int(xgb_probs[i] >= 0.5),
            "xgb_prob": float(xgb_probs[i]),
            "up_ask": up_asks[i],
            "down_ask": down_asks[i],
        }
        for i in range(len(snap_rows))
    ]
    all_cd.append({"candle_id": cid, "truth": truth, "snapshots": sd})

print(f"Built predictions for {len(all_cd)} candles")

## 5. Scaling-in engine

In [ ]:
def run_scaling(name, entry_points, model_key="lr", bet_per_entry=10.0, min_confidence=0.0):
    pred_key = f"{model_key}_pred"
    prob_key = f"{model_key}_prob"
    bal = 1000.0
    history = [bal]
    total_bets, total_wins, candles_entered, candles_skipped = 0, 0, 0, 0

    for cd in all_cd:
        sd = cd["snapshots"]
        truth = cd["truth"]
        entries = []
        first_direction = None

        for min_e, n_c in entry_points:
            for i in range(max(n_c - 1, 0), len(sd)):
                if sd[i]["elapsed_pct"] < min_e:
                    continue
                if any(i <= prev_tick for prev_tick, _, _ in entries):
                    continue
                if n_c > 1 and not all(sd[i - j][pred_key] == sd[i][pred_key] for j in range(n_c)):
                    continue
                confidence = max(sd[i][prob_key], 1.0 - sd[i][prob_key])
                if confidence < min_confidence:
                    continue
                direction = sd[i][pred_key]
                if first_direction is None:
                    first_direction = direction
                elif direction != first_direction:
                    break
                ask = sd[i]["up_ask"] if direction == 1 else sd[i]["down_ask"]
                if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                    continue
                entries.append((i, direction, ask))
                break

        if not entries:
            candles_skipped += 1
            continue
        candles_entered += 1
        for _, direction, ask in entries:
            if bal < bet_per_entry:
                break
            total_bets += 1
            if direction == truth:
                bal += (bet_per_entry / ask) * (1.0 - ask)
                total_wins += 1
            else:
                bal -= bet_per_entry
        history.append(bal)

    wr = total_wins / total_bets if total_bets > 0 else 0
    max_dd = (
        max((p - h) / p for p, h in zip([max(history[: i + 1]) for i in range(len(history))], history, strict=False))
        if len(history) > 1
        else 0
    )
    return {
        "name": name,
        "balance": bal,
        "history": history,
        "total_bets": total_bets,
        "candles_entered": candles_entered,
        "wins": total_wins,
        "win_rate": wr,
        "return": (bal - 1000) / 1000 * 100,
        "max_dd": max_dd,
    }

## 6. Run strategies

In [ ]:
strategies = [
    # LR strategies
    ("LR 1x e5%", [(0.05, 3)], "lr", 0.0),
    ("LR 1x e5% conf>0.7", [(0.05, 3)], "lr", 0.7),
    ("LR 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "lr", 0.0),
    ("LR 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "lr", 0.6),
    ("LR 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "lr", 0.7),
    # RF strategies
    ("RF 1x e5%", [(0.05, 3)], "rf", 0.0),
    ("RF 1x e5% conf>0.7", [(0.05, 3)], "rf", 0.7),
    ("RF 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "rf", 0.0),
    ("RF 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "rf", 0.6),
    ("RF 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "rf", 0.7),
    # XGBoost strategies
    ("XGB 1x e5%", [(0.05, 3)], "xgb", 0.0),
    ("XGB 1x e5% conf>0.6", [(0.05, 3)], "xgb", 0.6),
    ("XGB 1x e5% conf>0.7", [(0.05, 3)], "xgb", 0.7),
    ("XGB 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "xgb", 0.0),
    ("XGB 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "xgb", 0.6),
    ("XGB 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "xgb", 0.7),
    # RF delayed entry
    ("RF 1x e30%", [(0.30, 3)], "rf", 0.0),
    ("RF 2x e30%+e60% conf>0.6", [(0.30, 3), (0.60, 3)], "rf", 0.6),
]

results = []
print(f"{'Strategy':<30} {'Bets':>5} {'WR':>6} {'Balance':>10} {'Return':>8} {'MaxDD':>7}")
print("-" * 72)

for name, eps, model_key, conf in strategies:
    r = run_scaling(name, eps, model_key=model_key, min_confidence=conf)
    results.append(r)
    print(
        f"{r['name']:<30} {r['total_bets']:>5} "
        f"{r['win_rate'] * 100:>5.1f}% ${r['balance']:>9.2f} {r['return']:>+7.1f}% {r['max_dd'] * 100:>6.1f}%"
    )

## 7. Equity curves — LR vs RF vs XGBoost

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

colors = {"LR": "#3498db", "RF": "#e74c3c", "XGB": "#e67e22"}

# Top: all strategies
for r in results:
    prefix = r["name"].split(" ")[0]
    axes[0].plot(r["history"], color=colors.get(prefix, "gray"), alpha=0.3, linewidth=1)
for prefix, color in colors.items():
    axes[0].plot([], [], color=color, label=f"{prefix} strategies", linewidth=2)
axes[0].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Candle #")
axes[0].set_ylabel("Balance ($)")
axes[0].set_title("All Strategies — LR (blue) vs RF (red) vs XGB (orange)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bottom: best of each
best = {}
for r in results:
    prefix = r["name"].split(" ")[0]
    if prefix not in best or r["balance"] > best[prefix]["balance"]:
        best[prefix] = r

for prefix in ["LR", "RF", "XGB"]:
    if prefix in best:
        r = best[prefix]
        axes[1].plot(
            r["history"],
            color=colors[prefix],
            linewidth=2.5,
            label=f"{r['name']} -> ${r['balance']:,.0f} ({r['return']:+.1f}%, WR={r['win_rate'] * 100:.0f}%)",
        )

axes[1].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Candle #")
axes[1].set_ylabel("Balance ($)")
axes[1].set_title("Best LR vs Best RF vs Best XGB")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 92)
print("LR vs RF vs XGBoost — FORWARD TEST SUMMARY")
print(f"Candles: {len(all_cd)} (unseen during training)")
print("Models loaded from: models/ (pre-trained, no re-training)")
print("=" * 92)

print(f"\n{'':30} {'LR best':>20} {'RF best':>20} {'XGB best':>20}")
print("-" * 92)
for metric, key in [
    ("Strategy", "name"),
    ("Win Rate", "win_rate"),
    ("Return", "return"),
    ("Max Drawdown", "max_dd"),
    ("Final Balance", "balance"),
    ("Total Bets", "total_bets"),
]:
    vals = []
    for prefix in ["LR", "RF", "XGB"]:
        r = best.get(prefix, {})
        v = r.get(key, "N/A")
        if key == "win_rate":
            vals.append(f"{v * 100:.1f}%" if isinstance(v, float) else str(v))
        elif key == "return":
            vals.append(f"{v:+.1f}%" if isinstance(v, int | float) else str(v))
        elif key == "max_dd":
            vals.append(f"{v * 100:.1f}%" if isinstance(v, float) else str(v))
        elif key == "balance":
            vals.append(f"${v:,.2f}" if isinstance(v, float) else str(v))
        else:
            vals.append(str(v))
    print(f"{metric:<30} {vals[0]:>20} {vals[1]:>20} {vals[2]:>20}")

balances = [(prefix, best[prefix]["balance"]) for prefix in ["LR", "RF", "XGB"] if prefix in best]
winner = max(balances, key=lambda x: x[1])[0]
print(f"\nWinner: {winner}")
print("\nNote: Small sample size — results are directional, not conclusive.")
print("Re-run after collecting more data for robust comparison.")

## 9. Conclusion

All three models loaded from `models/` (no re-training). Tested on candles never seen during training.

Re-run as more data accumulates. Export updated models via notebooks 10, 13, 15.
